# 🗄️ Export PostgreSQL → JSON for Colab

**Chạy notebook này LOCAL** (máy có kết nối DB) trước khi mở Colab.  

```
Workflow:
  [LOCAL] Notebook này          →  train.json + test.json
                                        ↓
  [COLAB] tft_colab.ipynb      ←  Upload JSON lên Drive
  [COLAB] gbm_ensemble_colab   ←  Upload JSON lên Drive
```

> ⚠️ Yêu cầu: PostgreSQL đang chạy + `.env` đã cấu hình đúng

## ⚙️ Section 0 — Cấu hình

In [3]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent  # giả sử notebook nằm trong notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

# ── Tham số export ───────────────────────────────────────────────────────────
SYMBOL      = 'BTC/USDT'    # Symbol chính
TIMEFRAME   = '1d'          # Khung thời gian: '1d', '4h', '1h', ...
LOOKBACK_DAYS = 730         # Lookback 2 năm
TEST_RATIO  = 0.15          # 15% cuối làm test
OUTPUT_DIR  = '../colab_data'  # Thư mục lưu JSON

# Nhiều symbols (bỏ comment nếu muốn load nhiều symbol)
# SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT']
SYMBOLS = None

print(f'Config:')
print(f'  Symbol      = {SYMBOLS or SYMBOL}')
print(f'  Timeframe   = {TIMEFRAME}')
print(f'  Lookback    = {LOOKBACK_DAYS} days')
print(f'  Test ratio  = {TEST_RATIO:.0%}')
print(f'  Output      = {OUTPUT_DIR}/')

Config:
  Symbol      = BTC/USDT
  Timeframe   = 1d
  Lookback    = 730 days
  Test ratio  = 15%
  Output      = ../colab_data/


## 🔌 Section 1 — Kiểm tra kết nối DB

In [4]:
# Kiểm tra .env và DB config
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / '.env')

print('DB Config từ .env:')
print(f"  DB_HOST = {os.getenv('DB_HOST', 'NOT SET')}")
print(f"  DB_PORT = {os.getenv('DB_PORT', 'NOT SET')}")
print(f"  DB_NAME = {os.getenv('DB_NAME', 'NOT SET')}")
print(f"  DB_USER = {os.getenv('DB_USER', 'NOT SET')}")
print(f"  DB_PASS = {'*****' if os.getenv('DB_PASSWORD') else 'NOT SET'}")

DB Config từ .env:
  DB_HOST = localhost
  DB_PORT = 5432
  DB_NAME = financial_ts
  DB_USER = postgres
  DB_PASS = *****


In [5]:
# Test kết nối
from models.data_loader import OHLCVDBLoader

try:
    loader = OHLCVDBLoader()
    print('✅ Kết nối PostgreSQL thành công!')
except Exception as e:
    print(f'❌ Lỗi kết nối: {e}')
    print('\n💡 Kiểm tra:')
    print('  1. PostgreSQL đang chạy?')
    print('  2. .env đã có DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD?')
    print('  3. Database đã có data (chạy run_ingestion.py trước)?')

2026-08-17 10:02:22.129 | SUCCESS  | data_collection_api.db_manager:_create_engine:48 - Connected to PostgreSQL: localhost:5432/financial_ts
2026-08-17 10:02:22.192 | INFO     | data_collection_api.db_manager:_apply_schema:99 - Schema applied (tables, indexes & triggers ready).
2026-08-17 10:02:22.192 | INFO     | models.data_loader:__init__:89 - OHLCVDBLoader: kết nối PostgreSQL thành công.


✅ Kết nối PostgreSQL thành công!


## 📊 Section 2 — Xem data có sẵn trong DB

In [6]:
import pandas as pd

# Load thử 5 rows để xem cấu trúc
df_preview = loader.load(
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    lookback_days=30,   # chỉ load 30 ngày để preview nhanh
)

print(f'\nShape: {df_preview.shape}')
print(f'Index: {df_preview.index.name} ({df_preview.index.dtype})')
print(f'Columns ({len(df_preview.columns)}):')
for i, col in enumerate(df_preview.columns):
    print(f'  {i+1:2}. {col}')

print('\nSample data:')
display(df_preview.tail(3))

2026-08-17 10:02:22.202 | INFO     | models.data_loader:load:120 - Loading BTC/USDT 1d | lookback=30d ...
2026-08-17 10:02:22.226 | INFO     | eda.data:load:64 - Loaded 29 rows | BTC/USDT 1d | 2026-07-19 → 2026-08-16
2026-08-17 10:02:22.247 | INFO     | models.data_loader:load:129 - Loaded: 10 rows | 2026-08-07 → 2026-08-16 | 45 columns



Shape: (10, 45)
Index: open_time (datetime64[us, UTC])
Columns (45):
   1. open
   2. high
   3. low
   4. close
   5. volume
   6. return_pct
   7. log_return
   8. ma_7
   9. ma_25
  10. ma_99
  11. ema_12
  12. ema_26
  13. macd
  14. macd_signal
  15. macd_hist
  16. rsi_14
  17. bb_mid
  18. bb_upper
  19. bb_lower
  20. bb_width
  21. bb_pct
  22. atr_14
  23. volume_ma_20
  24. volume_ratio
  25. rolling_vol_30
  26. price_zscore
  27. drawdown_pct
  28. regime
  29. vol_regime
  30. day_of_week
  31. hour
  32. month
  33. year
  34. day_of_month
  35. day_of_year
  36. hour_sin
  37. hour_cos
  38. dow_sin
  39. dow_cos
  40. month_sin
  41. month_cos
  42. dom_sin
  43. dom_cos
  44. doy_sin
  45. doy_cos

Sample data:


,open,high,low,close,volume,return_pct,log_return,ma_7,ma_25,ma_99,...,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,dom_sin,dom_cos,doy_sin,doy_cos
open_time,,,,,,,,,,,,,,,,,,,,,
2026-08-14 00:00:00+00:00,63490.86,63617.45,62535.24,63043.56,12340.82899,-0.704511,-0.007070,63921.230000,64255.5392,NaN,...,0.0,1.0,-0.433884,-0.900969,-0.5,-0.866025,0.485302,-0.874347,-0.666089,-0.745872
2026-08-15 00:00:00+00:00,63043.56,63187.98,62920.00,63086.01,5405.16038,0.067334,0.000673,63653.145714,64116.7332,NaN,...,0.0,1.0,-0.974928,-0.222521,-0.5,-0.866025,0.299363,-0.954139,-0.678820,-0.734304
2026-08-16 00:00:00+00:00,63086.01,63158.80,63012.00,63054.83,1281.88334,-0.049425,-0.000494,63389.322857,63994.3468,NaN,...,0.0,1.0,-0.781831,0.623490,-0.5,-0.866025,0.101168,-0.994869,-0.691351,-0.722519


## 💾 Section 3 — Export train/test JSON

In [7]:
# ── Export JSON ──────────────────────────────────────────────────────────────
result = loader.export_for_colab(
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    lookback_days=LOOKBACK_DAYS,
    output_dir=OUTPUT_DIR,
    test_ratio=TEST_RATIO,
    symbols=SYMBOLS,        # None nếu chỉ 1 symbol
)

print('\n✅ Export xong!')
print(f"  train.json : {result['metadata']['train_rows']:,} rows")
print(f"  test.json  : {result['metadata']['test_rows']:,} rows")

loader.close()

2026-08-17 10:02:22.273 | INFO     | models.data_loader:load:120 - Loading BTC/USDT 1d | lookback=730d ...
2026-08-17 10:02:22.286 | INFO     | eda.data:load:64 - Loaded 729 rows | BTC/USDT 1d | 2024-08-18 → 2026-08-16
2026-08-17 10:02:22.308 | INFO     | models.data_loader:load:129 - Loaded: 710 rows | 2024-09-06 → 2026-08-16 | 45 columns
2026-08-17 10:02:22.368 | INFO     | models.data_loader:export_for_colab:304 - ============================================================
2026-08-17 10:02:22.369 | INFO     | models.data_loader:export_for_colab:305 - Export hoàn tất!
2026-08-17 10:02:22.369 | INFO     | models.data_loader:export_for_colab:306 -   train.json    : 603 rows → ..\colab_data\train.json
2026-08-17 10:02:22.370 | INFO     | models.data_loader:export_for_colab:307 -   test.json     : 107 rows → ..\colab_data\test.json
2026-08-17 10:02:22.370 | INFO     | models.data_loader:export_for_colab:308 -   metadata.json : → ..\colab_data\metadata.json
2026-08-17 10:02:22.370 | INFO


✅ Export xong!
  train.json : 603 rows
  test.json  : 107 rows


In [8]:
# ── Verify JSON files ─────────────────────────────────────────────────────────
import json

with open(result['train_path']) as f:
    train_sample = json.load(f)

print(f'Train JSON: {len(train_sample):,} records')
print(f'Keys: {list(train_sample[0].keys())}')
print(f'\nFirst record:')
import json as js
print(js.dumps(train_sample[0], indent=2, default=str)[:500])

Train JSON: 603 records
Keys: ['open_time', 'open', 'high', 'low', 'close', 'volume', 'return_pct', 'log_return', 'ma_7', 'ma_25', 'ma_99', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist', 'rsi_14', 'bb_mid', 'bb_upper', 'bb_lower', 'bb_width', 'bb_pct', 'atr_14', 'volume_ma_20', 'volume_ratio', 'rolling_vol_30', 'price_zscore', 'drawdown_pct', 'regime', 'vol_regime', 'day_of_week', 'hour', 'month', 'year', 'day_of_month', 'day_of_year', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'dom_sin', 'dom_cos', 'doy_sin', 'doy_cos', 'symbol']

First record:
{
  "open_time": "2024-09-06 00:00:00+00:00",
  "open": 56180.0,
  "high": 57008.0,
  "low": 52550.0,
  "close": 53962.97,
  "volume": 54447.76826,
  "return_pct": -3.9462976148095352,
  "log_return": -0.040262751080288636,
  "ma_7": 57287.08285714286,
  "ma_25": NaN,
  "ma_99": NaN,
  "ema_12": 57919.883978377766,
  "ema_26": 58732.129802637675,
  "macd": -812.2458242599096,
  "macd_signal": -127.239116018

## 📋 Section 4 — Metadata

In [9]:
import json

with open(result['meta_path']) as f:
    meta = json.load(f)

print('── Metadata ────────────────────────────────────')
for k, v in meta.items():
    if isinstance(v, dict):
        print(f'  {k}:')
        for kk, vv in v.items():
            print(f'    {kk}: {vv}')
    elif isinstance(v, list) and len(v) > 5:
        print(f'  {k}: [{v[0]}, {v[1]}, ..., {v[-1]}] ({len(v)} items)')
    else:
        print(f'  {k}: {v}')

── Metadata ────────────────────────────────────
  symbol: BTC/USDT
  timeframe: 1d
  lookback_days: 730
  test_ratio: 0.15
  total_rows: 710
  train_rows: 603
  test_rows: 107
  columns: [open_time, open, ..., symbol] (47 items)
  n_features: 47
  date_range:
    start: 2024-09-06 00:00:00+00:00
    end: 2026-08-16 00:00:00+00:00
    train_end: 2026-05-01 00:00:00+00:00
    test_start: 2026-05-02 00:00:00+00:00
  exported_at: 2026-08-17T03:02:22.367815+00:00


## 🚀 Section 5 — Bước tiếp theo

Files đã sẵn sàng tại `colab_data/`:

```
colab_data/
├── train.json      ← dùng cho TFT và GBM notebooks
├── test.json       ← dùng cho TFT và GBM notebooks
└── metadata.json   ← thông tin về data
```

**Upload lên Google Drive:**
1. Vào [drive.google.com](https://drive.google.com)
2. Tạo folder `Financial-Forecast/data/`
3. Upload `train.json` và `test.json`

**Mở Colab notebooks:**
- [`tft_colab.ipynb`](tft_colab.ipynb) — TFT model
- [`gbm_ensemble_colab.ipynb`](gbm_ensemble_colab.ipynb) — XGBoost + LightGBM

**Trong notebook Colab**, chỉnh:**
```python
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

TRAIN_JSON = '/content/drive/MyDrive/Financial-Forecast/data/train.json'
TEST_JSON  = '/content/drive/MyDrive/Financial-Forecast/data/test.json'
```